# 05 - Análise Financeira para o Dashboard CTI

Este notebook complementa os notebooks `03_kpis.ipynb` e `04_analise.ipynb`, sem recalcular os indicadores ou repetir os gráficos já produzidos. O foco é preparar **variações por cenário, alertas financeiros, comparações direcionadas e simulações de sensibilidade WACC/EVA**, com tabelas exportáveis para o dashboard em Streamlit.

**Fonte:** aba `_Indicadores` da planilha `Analise_Financeira.xlsx`, gerada na etapa de KPIs. Os cenários são simulações, não resultados efetivamente realizados pela empresa. Não são criadas metas fictícias.

## 1. Importação das bibliotecas e localização da base

Carregamos os resultados consolidados dos KPIs diretamente da planilha existente. O código encontra a raiz do projeto independentemente de o notebook ser executado a partir da pasta `notebooks` ou da raiz.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

PASTA_ATUAL = Path.cwd().resolve()
RAIZ = next(
    (p for p in [PASTA_ATUAL, *PASTA_ATUAL.parents]
     if (p / "notebooks").is_dir() and (p / "documentos").is_dir()),
    None
)
if RAIZ is None:
    raise FileNotFoundError("Execute o notebook dentro da pasta do projeto CTI.")

PLANILHA = RAIZ / "documentos" / "entrega 1" / "contabilidade e finanças" / "Analise_Financeira.xlsx"
if not PLANILHA.exists():
    raise FileNotFoundError(f"Planilha não encontrada: {PLANILHA}")

df = pd.read_excel(PLANILHA, sheet_name="_Indicadores")
print("Linhas e colunas:", df.shape)
display(df.head())

Linhas e colunas: (14400, 18)


,Cenario,Ano,Margem EBITDA,Margem Líquida,LC,LS,LI,LG,PCT,CE,IPL,NOPAT,Capital Investido,ROIC,WACC,EVA,MVA Aproximado,EBITDA
0,Total Cen_00001,Ano 1,0.829973,0.390840,1.661277,1.661197,1.115028,0.282561,4.277473,0.125446,4.068824,2.812862e+06,6.737529e+06,0.417492,0.08,2.273859e+06,2.842324e+07,5.188381e+06
1,Total Cen_00001,Ano 2,0.825317,0.356584,1.154874,1.154811,0.753056,0.278254,6.085068,0.152663,5.391876,2.324825e+06,6.416736e+06,0.362306,0.08,1.811486e+06,2.264357e+07,4.630952e+06
2,Total Cen_00001,Ano 3,0.853486,0.371077,1.301103,1.301054,0.992276,0.376489,6.137439,0.189511,4.826759,2.428926e+06,5.785050e+06,0.419863,0.08,1.966122e+06,2.457653e+07,4.883290e+06
3,Total Cen_00001,Ano 4,0.865342,0.389656,3.324041,3.323867,2.243316,0.376906,4.982482,0.062000,4.104556,2.554873e+06,6.983079e+06,0.365866,0.08,1.996226e+06,2.495283e+07,5.148066e+06
4,Total Cen_00001,Ano 5,0.871756,0.422448,0.664478,0.664409,0.243336,0.250630,4.902671,0.143564,3.266220,2.818760e+06,9.176816e+06,0.307161,0.08,2.084614e+06,2.605768e+07,5.453240e+06


## 2. Validação da estrutura e ordenação dos períodos

Verificamos as colunas necessárias, a unicidade de cada combinação cenário/ano e a disponibilidade de dados. O **Ano 12** apresenta ausências estruturais em alguns indicadores; esses valores permanecem ausentes e não são substituídos por zero.

In [2]:
obrigatorias = ["Cenario", "Ano", "Margem EBITDA", "Margem Líquida", "EBITDA", "NOPAT", "Capital Investido", "WACC", "EVA"]
faltantes = [c for c in obrigatorias if c not in df.columns]
if faltantes:
    raise ValueError(f"Colunas ausentes: {faltantes}")

if df.duplicated(["Cenario", "Ano"]).any():
    raise ValueError("Existem combinações duplicadas de cenário e ano.")

df["Ano_num"] = pd.to_numeric(df["Ano"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
if df["Ano_num"].isna().any():
    raise ValueError("Existem anos que não puderam ser interpretados.")

df["Ano_num"] = df["Ano_num"].astype(int)
colunas_numericas = df.select_dtypes(include="number").columns.difference(["Ano_num"])
df[colunas_numericas] = df[colunas_numericas].replace([np.inf, -np.inf], np.nan)
df = df.sort_values(["Cenario", "Ano_num"]).reset_index(drop=True)

qualidade = pd.DataFrame({
    "Indicador": colunas_numericas,
    "Disponíveis": [df[c].notna().sum() for c in colunas_numericas],
    "Ausentes": [df[c].isna().sum() for c in colunas_numericas],
})
print("Cenários:", df["Cenario"].nunique(), "| Anos:", df["Ano_num"].nunique())
display(qualidade.sort_values("Ausentes", ascending=False))

Cenários: 1200 | Anos: 12


,Indicador,Disponíveis,Ausentes
0,CE,13200,1200
1,Capital Investido,13200,1200
3,EVA,13200,1200
4,IPL,13200,1200
6,LG,13200,1200
14,ROIC,13200,1200
9,MVA Aproximado,13200,1200
8,LS,13200,1200
13,PCT,13200,1200
7,LI,14400,0


## 3. Variação anual dentro de cada cenário

O comparativo anual anterior utiliza medidas agregadas. Aqui calculamos a **variação de cada cenário entre anos consecutivos**, permitindo que o dashboard mostre o desempenho de um cenário escolhido pelo usuário. A variação percentual só é calculada quando existe uma base anterior diferente de zero.

In [3]:
metricas_variacao = ["EBITDA", "NOPAT", "EVA", "Margem EBITDA", "Margem Líquida"]
variacoes = df[["Cenario", "Ano", "Ano_num", *metricas_variacao]].copy()

for metrica in metricas_variacao:
    anterior = variacoes.groupby("Cenario")[metrica].shift(1)
    variacoes[f"{metrica} - Variação Absoluta"] = variacoes[metrica] - anterior
    variacoes[f"{metrica} - Variação %"] = np.where(
        anterior.notna() & anterior.ne(0),
        (variacoes[metrica] - anterior) / anterior.abs() * 100,
        np.nan
    )

print("Variações por cenário e ano:")
display(variacoes.head(15))

Variações por cenário e ano:


,Cenario,Ano,Ano_num,EBITDA,NOPAT,EVA,Margem EBITDA,Margem Líquida,EBITDA - Variação Absoluta,EBITDA - Variação %,NOPAT - Variação Absoluta,NOPAT - Variação %,EVA - Variação Absoluta,EVA - Variação %,Margem EBITDA - Variação Absoluta,Margem EBITDA - Variação %,Margem Líquida - Variação Absoluta,Margem Líquida - Variação %
0,Total Cen_00001,Ano 1,1,5.188381e+06,2.812862e+06,2.273859e+06,0.829973,0.390840,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Total Cen_00001,Ano 2,2,4.630952e+06,2.324825e+06,1.811486e+06,0.825317,0.356584,-5.574283e+05,-10.743783,-4.880371e+05,-17.350197,-462373.637840,-20.334310,-0.004656,-0.560950,-0.034256,-8.764710
2,Total Cen_00001,Ano 3,3,4.883290e+06,2.428926e+06,1.966122e+06,0.853486,0.371077,2.523377e+05,5.448938,1.041016e+05,4.477824,154636.460150,8.536443,0.028169,3.413108,0.014493,4.064384
3,Total Cen_00001,Ano 4,4,5.148066e+06,2.554873e+06,1.996226e+06,0.865342,0.389656,2.647761e+05,5.422084,1.259464e+05,5.185269,30104.048621,1.531138,0.011856,1.389094,0.018579,5.006747
4,Total Cen_00001,Ano 5,5,5.453240e+06,2.818760e+06,2.084614e+06,0.871756,0.422448,3.051736e+05,5.927927,2.638870e+05,10.328772,88388.022228,4.427755,0.006414,0.741226,0.032792,8.415640
5,Total Cen_00001,Ano 6,6,5.834764e+06,3.055476e+06,2.233772e+06,0.878105,0.444770,3.815243e+05,6.996288,2.367167e+05,8.397902,149157.585906,7.155164,0.006349,0.728351,0.022322,5.283973
6,Total Cen_00001,Ano 7,7,6.236914e+06,3.224144e+06,2.333587e+06,0.884270,0.447076,4.021502e+05,6.892313,1.686675e+05,5.520172,99814.650176,4.468435,0.006165,0.702050,0.002306,0.518580
7,Total Cen_00001,Ano 8,8,6.693782e+06,3.481231e+06,2.622628e+06,0.888241,0.454888,4.568678e+05,7.325221,2.570869e+05,7.973805,289041.194114,12.386135,0.003970,0.449010,0.007812,1.747338
8,Total Cen_00001,Ano 9,9,7.136917e+06,3.731434e+06,2.867592e+06,0.892089,0.466832,4.431346e+05,6.620093,2.502034e+05,7.187211,244963.945923,9.340401,0.003849,0.433273,0.011944,2.625615
9,Total Cen_00001,Ano 10,10,7.653142e+06,4.150515e+06,3.254978e+06,0.897302,0.496918,5.162255e+05,7.233173,4.190811e+05,11.231101,387386.501900,13.509123,0.005213,0.584328,0.030086,6.444714


## 4. Identificação de situações que merecem atenção

Criamos **sinalizadores objetivos**, sem inventar limites arbitrários: EVA negativo (destruição de valor pela premissa adotada), margem líquida negativa e queda do EBITDA em relação ao ano anterior. São alertas exploratórios, não diagnósticos automáticos da situação da CTI.

In [4]:
alertas = df[["Cenario", "Ano", "Ano_num", "EVA", "Margem Líquida", "EBITDA"]].copy()
alertas["EBITDA anterior"] = alertas.groupby("Cenario")["EBITDA"].shift(1)
alertas["EVA negativo"] = alertas["EVA"].lt(0).where(alertas["EVA"].notna())
alertas["Margem líquida negativa"] = alertas["Margem Líquida"].lt(0).where(alertas["Margem Líquida"].notna())
alertas["Queda do EBITDA"] = (
    alertas["EBITDA"].lt(alertas["EBITDA anterior"])
    .where(alertas["EBITDA"].notna() & alertas["EBITDA anterior"].notna())
)

colunas_alerta = ["EVA negativo", "Margem líquida negativa", "Queda do EBITDA"]
resumo_alertas = alertas.groupby(["Ano_num", "Ano"], as_index=False).agg(
    **{f"Casos - {c}": (c, "sum") for c in colunas_alerta},
    **{f"Observações válidas - {c}": (c, "count") for c in colunas_alerta}
).sort_values("Ano_num")

for c in colunas_alerta:
    resumo_alertas[f"Percentual - {c}"] = np.where(
        resumo_alertas[f"Observações válidas - {c}"] > 0,
        100 * resumo_alertas[f"Casos - {c}"].astype(float) / resumo_alertas[f"Observações válidas - {c}"].replace(0, np.nan).astype(float),
        np.nan
    )

display(resumo_alertas)

,Ano_num,Ano,Casos - EVA negativo,Casos - Margem líquida negativa,Casos - Queda do EBITDA,Observações válidas - EVA negativo,Observações válidas - Margem líquida negativa,Observações válidas - Queda do EBITDA,Percentual - EVA negativo,Percentual - Margem líquida negativa,Percentual - Queda do EBITDA
0,1,Ano 1,0,0,0,1200,1200,0,0.0,0.0,NaN
1,2,Ano 2,0,0,1200,1200,1200,1200,0.0,0.0,100.0
2,3,Ano 3,0,0,0,1200,1200,1200,0.0,0.0,0.0
3,4,Ano 4,0,0,0,1200,1200,1200,0.0,0.0,0.0
4,5,Ano 5,0,0,0,1200,1200,1200,0.0,0.0,0.0
5,6,Ano 6,0,0,0,1200,1200,1200,0.0,0.0,0.0
6,7,Ano 7,0,0,0,1200,1200,1200,0.0,0.0,0.0
7,8,Ano 8,0,0,0,1200,1200,1200,0.0,0.0,0.0
8,9,Ano 9,0,0,0,1200,1200,1200,0.0,0.0,0.0
9,10,Ano 10,0,0,0,1200,1200,1200,0.0,0.0,0.0


## 5. Comparação direcionada entre cenários

Em vez de repetir o ranking geral, preparamos uma função que retorna os indicadores de **até três cenários escolhidos** para um ano específico. Essa tabela poderá alimentar a página de comparação do Streamlit.

In [5]:
indicadores_comparacao = [
    "Margem EBITDA", "Margem Líquida", "LC", "LS", "LI", "LG",
    "PCT", "CE", "IPL", "NOPAT", "Capital Investido", "ROIC", "EVA"
]
indicadores_comparacao = [c for c in indicadores_comparacao if c in df.columns]

def comparar_cenarios(cenarios, ano):
    cenarios = list(dict.fromkeys(cenarios))
    if not 1 <= len(cenarios) <= 3:
        raise ValueError("Selecione de um a três cenários.")
    desconhecidos = set(cenarios) - set(df["Cenario"])
    if desconhecidos:
        raise ValueError(f"Cenários não encontrados: {sorted(desconhecidos)}")
    resultado = df.loc[
        df["Cenario"].isin(cenarios) & df["Ano_num"].eq(int(ano)),
        ["Cenario", "Ano", *indicadores_comparacao]
    ].copy()
    return resultado.set_index("Cenario").reindex(cenarios).reset_index()

cenarios_exemplo = df["Cenario"].drop_duplicates().head(3).tolist()
comparacao_exemplo = comparar_cenarios(cenarios_exemplo, ano=1)
display(comparacao_exemplo)

,Cenario,Ano,Margem EBITDA,Margem Líquida,LC,LS,LI,LG,PCT,CE,IPL,NOPAT,Capital Investido,ROIC,EVA
0,Total Cen_00001,Ano 1,0.829973,0.390840,1.661277,1.661197,1.115028,0.282561,4.277473,0.125446,4.068824,2.812862e+06,6.737529e+06,0.417492,2.273859e+06
1,Total Cen_00002,Ano 1,0.829762,0.390877,1.656626,1.656546,1.109714,0.281926,4.291442,0.125392,4.081573,2.802863e+06,6.736811e+06,0.416052,2.263918e+06
2,Total Cen_00003,Ano 1,0.829394,0.392178,1.655928,1.655848,1.105759,0.282011,4.295743,0.125496,4.084297,2.788191e+06,6.735535e+06,0.413952,2.249348e+06


## 6. Sensibilidade do EVA à taxa WACC

O notebook de KPIs adota **WACC de 8%** como premissa. Nesta análise, preservamos o NOPAT e o Capital Investido originais e recalculamos somente o EVA sob diferentes taxas: `EVA simulado = NOPAT − Capital Investido × WACC simulado`. O intervalo abaixo é ilustrativo e pode ser ajustado no dashboard. Não representa taxas de mercado observadas.

In [6]:
taxas_wacc = np.round(np.arange(0.04, 0.1201, 0.01), 4)
base_sensibilidade = df[["Cenario", "Ano", "Ano_num", "NOPAT", "Capital Investido", "WACC", "EVA"]].copy()
base_sensibilidade = base_sensibilidade.dropna(subset=["NOPAT", "Capital Investido"])

sensibilidade = base_sensibilidade.merge(
    pd.DataFrame({"WACC simulado": taxas_wacc}), how="cross"
)
sensibilidade["EVA simulado"] = (
    sensibilidade["NOPAT"]
    - sensibilidade["Capital Investido"] * sensibilidade["WACC simulado"]
)
sensibilidade["Diferença para EVA original"] = (
    sensibilidade["EVA simulado"] - sensibilidade["EVA"]
)

print("Registros simulados:", len(sensibilidade))
display(sensibilidade.head(12))

Registros simulados: 118800


,Cenario,Ano,Ano_num,NOPAT,Capital Investido,WACC,EVA,WACC simulado,EVA simulado,Diferença para EVA original
0,Total Cen_00001,Ano 1,1,2.812862e+06,6.737529e+06,0.08,2.273859e+06,0.04,2.543361e+06,269501.170855
1,Total Cen_00001,Ano 1,1,2.812862e+06,6.737529e+06,0.08,2.273859e+06,0.05,2.475985e+06,202125.878141
2,Total Cen_00001,Ano 1,1,2.812862e+06,6.737529e+06,0.08,2.273859e+06,0.06,2.408610e+06,134750.585428
3,Total Cen_00001,Ano 1,1,2.812862e+06,6.737529e+06,0.08,2.273859e+06,0.07,2.341235e+06,67375.292714
4,Total Cen_00001,Ano 1,1,2.812862e+06,6.737529e+06,0.08,2.273859e+06,0.08,2.273859e+06,0.000000
5,Total Cen_00001,Ano 1,1,2.812862e+06,6.737529e+06,0.08,2.273859e+06,0.09,2.206484e+06,-67375.292714
6,Total Cen_00001,Ano 1,1,2.812862e+06,6.737529e+06,0.08,2.273859e+06,0.10,2.139109e+06,-134750.585428
7,Total Cen_00001,Ano 1,1,2.812862e+06,6.737529e+06,0.08,2.273859e+06,0.11,2.071734e+06,-202125.878141
8,Total Cen_00001,Ano 1,1,2.812862e+06,6.737529e+06,0.08,2.273859e+06,0.12,2.004358e+06,-269501.170855
9,Total Cen_00001,Ano 2,2,2.324825e+06,6.416736e+06,0.08,1.811486e+06,0.04,2.068155e+06,256669.452919


## 7. Ponto de equilíbrio econômico do WACC

Quando o Capital Investido é positivo, a taxa `NOPAT / Capital Investido` indica o WACC em que o EVA se torna zero. O resultado coincide matematicamente com o ROIC calculado sobre as mesmas grandezas. Aqui ele é utilizado como **limite de sensibilidade**, não como estimativa do custo de capital da empresa.

In [7]:
equilibrio = df[["Cenario", "Ano", "Ano_num", "NOPAT", "Capital Investido", "WACC", "EVA"]].copy()
equilibrio["WACC de equilíbrio"] = np.where(
    equilibrio["Capital Investido"].gt(0),
    equilibrio["NOPAT"] / equilibrio["Capital Investido"],
    np.nan
)
equilibrio["Folga para WACC atual (p.p.)"] = (
    equilibrio["WACC de equilíbrio"] - equilibrio["WACC"]
) * 100

display(equilibrio.head(12))

,Cenario,Ano,Ano_num,NOPAT,Capital Investido,WACC,EVA,WACC de equilíbrio,Folga para WACC atual (p.p.)
0,Total Cen_00001,Ano 1,1,2.812862e+06,6.737529e+06,0.08,2.273859e+06,0.417492,33.749159
1,Total Cen_00001,Ano 2,2,2.324825e+06,6.416736e+06,0.08,1.811486e+06,0.362306,28.230642
2,Total Cen_00001,Ano 3,3,2.428926e+06,5.785050e+06,0.08,1.966122e+06,0.419863,33.986263
3,Total Cen_00001,Ano 4,4,2.554873e+06,6.983079e+06,0.08,1.996226e+06,0.365866,28.586621
4,Total Cen_00001,Ano 5,5,2.818760e+06,9.176816e+06,0.08,2.084614e+06,0.307161,22.716096
5,Total Cen_00001,Ano 6,6,3.055476e+06,1.027130e+07,0.08,2.233772e+06,0.297477,21.747694
6,Total Cen_00001,Ano 7,7,3.224144e+06,1.113197e+07,0.08,2.333587e+06,0.289629,20.962933
7,Total Cen_00001,Ano 8,8,3.481231e+06,1.073254e+07,0.08,2.622628e+06,0.324362,24.436232
8,Total Cen_00001,Ano 9,9,3.731434e+06,1.079803e+07,0.08,2.867592e+06,0.345566,26.556617
9,Total Cen_00001,Ano 10,10,4.150515e+06,1.119421e+07,0.08,3.254978e+06,0.370773,29.077327


## 8. Resumo da sensibilidade para o dashboard

Consolidamos a mediana do EVA simulado por ano e taxa de WACC, além da proporção de cenários com EVA negativo. O resultado é útil para um gráfico de linhas ou um mapa de calor. A mediana é calculada apenas sobre os registros com dados suficientes.

In [8]:
sensibilidade["EVA negativo simulado"] = sensibilidade["EVA simulado"].lt(0)
resumo_sensibilidade = (
    sensibilidade
    .groupby(["Ano_num", "Ano", "WACC simulado"], as_index=False)
    .agg(
        EVA_mediano=("EVA simulado", "median"),
        EVA_medio=("EVA simulado", "mean"),
        Cenarios_validos=("Cenario", "nunique"),
        Percentual_EVA_negativo=("EVA negativo simulado", "mean")
    )
    .sort_values(["Ano_num", "WACC simulado"])
)
resumo_sensibilidade["Percentual_EVA_negativo"] *= 100
display(resumo_sensibilidade.head(15))

,Ano_num,Ano,WACC simulado,EVA_mediano,EVA_medio,Cenarios_validos,Percentual_EVA_negativo
0,1,Ano 1,0.04,2.515150e+06,2.515098e+06,1200,0.0
1,1,Ano 1,0.05,2.447809e+06,2.447134e+06,1200,0.0
2,1,Ano 1,0.06,2.380392e+06,2.379170e+06,1200,0.0
3,1,Ano 1,0.07,2.312998e+06,2.311206e+06,1200,0.0
4,1,Ano 1,0.08,2.245657e+06,2.243242e+06,1200,0.0
5,1,Ano 1,0.09,2.178289e+06,2.175277e+06,1200,0.0
6,1,Ano 1,0.10,2.110946e+06,2.107313e+06,1200,0.0
7,1,Ano 1,0.11,2.043593e+06,2.039349e+06,1200,0.0
8,1,Ano 1,0.12,1.976254e+06,1.971385e+06,1200,0.0
9,2,Ano 2,0.04,2.059241e+06,2.051605e+06,1200,0.0


## 9. Verificação da consistência do EVA original

Comparamos o EVA informado pela planilha com a fórmula aplicada à taxa WACC original, respeitando uma tolerância numérica. Diferenças inesperadas devem ser investigadas no `03_kpis.ipynb`, sem alterar silenciosamente os dados aqui.

In [9]:
conferencia_eva = df[["Cenario", "Ano", "NOPAT", "Capital Investido", "WACC", "EVA"]].dropna().copy()
conferencia_eva["EVA recalculado para conferência"] = (
    conferencia_eva["NOPAT"] - conferencia_eva["Capital Investido"] * conferencia_eva["WACC"]
)
conferencia_eva["Diferença absoluta"] = (
    conferencia_eva["EVA"] - conferencia_eva["EVA recalculado para conferência"]
).abs()
conferencia_eva["Consistente"] = np.isclose(
    conferencia_eva["EVA"],
    conferencia_eva["EVA recalculado para conferência"],
    rtol=1e-8,
    atol=0.01
)
print("Registros conferidos:", len(conferencia_eva))
print("Divergências:", (~conferencia_eva["Consistente"]).sum())
display(conferencia_eva.loc[~conferencia_eva["Consistente"]].head(20))

Registros conferidos: 13200
Divergências: 0


,Cenario,Ano,NOPAT,Capital Investido,WACC,EVA,EVA recalculado para conferência,Diferença absoluta,Consistente


## 10. Exportação dos resultados para o Streamlit

Salvamos arquivos CSV separados, com nomes claros e codificação UTF-8. Eles contêm somente **resultados derivados**, sem modificar os dados de origem ou as planilhas anteriores. O dashboard poderá carregá-los e aplicar filtros por cenário, ano e WACC.

In [10]:
SAIDA = RAIZ / "data" / "processed"
SAIDA.mkdir(parents=True, exist_ok=True)

arquivos_saida = {
    "variacoes_cenarios.csv": variacoes,
    "alertas_financeiros.csv": alertas,
    "resumo_alertas.csv": resumo_alertas,
    "comparacao_cenarios_base.csv": df[["Cenario", "Ano", "Ano_num", *indicadores_comparacao]],
    "sensibilidade_wacc_eva.csv": sensibilidade,
    "resumo_sensibilidade.csv": resumo_sensibilidade,
    "wacc_equilibrio.csv": equilibrio,
}

for nome, tabela in arquivos_saida.items():
    destino = SAIDA / nome
    tabela.to_csv(destino, index=False, sep=";", encoding="utf-8-sig")
    print(f"{nome}: {len(tabela):,} linhas")

print("Arquivos salvos em:", SAIDA)

variacoes_cenarios.csv: 14,400 linhas
alertas_financeiros.csv: 14,400 linhas
resumo_alertas.csv: 12 linhas
comparacao_cenarios_base.csv: 14,400 linhas
sensibilidade_wacc_eva.csv: 118,800 linhas
resumo_sensibilidade.csv: 99 linhas
wacc_equilibrio.csv: 14,400 linhas
Arquivos salvos em: C:\Users\julia\OneDrive - Etec Centro Paula Souza\Documents\GitHub\Projeto1\data\processed


## 11. Conclusão e utilização no dashboard

Este notebook acrescenta análises que não estavam consolidadas nos notebooks anteriores:

- **Tendências:** variações anuais por cenário, incluindo a diferença absoluta e percentual.
- **Insights:** sinalizadores de EVA negativo, margem líquida negativa e queda de EBITDA.
- **Comparação:** seleção de até três cenários em um ano específico.
- **Simulador:** EVA sob diferentes taxas WACC, ponto de equilíbrio e resumo de sensibilidade.
- **Integração:** arquivos CSV na pasta `data/processed`.

**Limitações:** os resultados representam cenários simulados; as taxas WACC testadas são hipóteses; valores ausentes não são imputados; e não há base confirmada para comparar metas com resultados realizados ou calcular ticket. Essas funcionalidades devem ser desenvolvidas somente após a definição e validação das fontes e premissas correspondentes.